# 06 · Embedding MLP (Step 3)

The brief's required deep-learning model: a compact MLP with **learned embeddings for airline, origin and destination**, on the same 40 numeric Phase 1 features, rows, splits and rolling folds as LogReg and XGBoost.

* **Network:** embeddings (width 1.6·n^0.56, at most 16) + numerics → 2–3 × [Linear → LayerNorm → ReLU → Dropout] → logit. LayerNorm, not BatchNorm, so a row's score never depends on the rest of its batch.
* **Loss:** binary cross-entropy, no class weights (calibrated probabilities are a deliverable).
* **Regularisation:** dropout, AdamW weight decay, early stopping on the **last 14 days of each fold's training window** (never the validation month).
* **Protocol:** 20 random configurations scored with one seed; the best is re-scored as a 5-seed ensemble (mean logit), which is the number used for model selection.

In [ ]:
import sys, json, pathlib
ROOT = pathlib.Path.cwd().resolve().parent
for p in (ROOT / 'src', ROOT.parent / 'phase1' / 'src'):
    sys.path.insert(0, str(p))

import aerocast_dl  # torch before XGBoost (OpenMP)
import numpy as np, pandas as pd
from IPython.display import Image, display
from aerocast_dl import config, pipeline
R, F = config.RESULTS_DIR, config.FIGURES_DIR
pd.set_option('display.precision', 4)


## 1. Search (single seed, rolling CV)

In [ ]:
search = pd.read_csv(R / 'search_mlp.csv').sort_values('cv_pr_auc', ascending=False)
best = pipeline.load_spec('mlp')
assert all(getattr(best, k) == search.iloc[0][k] for k in config.MLP_GRID)
search.head(8)

## 2. Five-seed ensemble against Phase 1

Same four folds (validation May, Jun, Jul, Aug), each model scored as deployed. The paired fold differences matter more than the means: the fold-to-fold spread (~0.07) is much larger than the differences between models.

In [ ]:
cv = pd.concat([pipeline.phase1_cv(), pd.read_csv(R / 'cv_models.csv')], ignore_index=True)
cv = cv[~cv['model'].str.startswith('ablation:')].set_index('model')
lr = cv.loc['Phase 1 LogReg', pipeline.FOLD_COLS].astype(float)
rows = ['Phase 1 LogReg', 'Phase 1 XGBoost', 'mlp']
t = cv.loc[rows, ['cv_pr_auc_mean', 'cv_pr_auc_std', *pipeline.FOLD_COLS, 'seed_sd_mean_over_folds', 'mean_best_epoch', 'n_params']]
d = cv.loc['mlp', pipeline.FOLD_COLS].astype(float) - lr
print(f'MLP - LogReg per fold: {np.round(d.to_numpy(), 4)} -> MLP better in {(d > 0).sum()}/4 folds')
t

## 3. Learning curves

Largest fold (validation August), search seed, trained for every epoch without early stopping. The inner-holdout loss bottoms out after a few epochs while the training loss keeps falling: early stopping, not the epoch budget, decides when the network stops.

In [ ]:
curves = json.loads((R / 'learning_curves.json').read_text())
for name, h in curves.items():
    es = [r['es_logloss'] for r in h]
    print(f"{name}: best inner-holdout epoch {int(np.argmin(es)) + 1}, loss {min(es):.4f} -> {es[-1]:.4f} at epoch {len(es)}")
display(Image(F / 'learning_curves.png', width=700))

## 4. What the embeddings learned

The final MLP's carrier and destination embeddings, projected on their first two principal components (seed 42). Neighbouring points are categories the network treats alike. This is descriptive only, not evidence for a causal grouping.

In [ ]:
for f in ['embedding_carrier.png', 'embedding_dest.png']:
    if (F / f).exists():
        display(Image(F / f, width=420))
    else:
        print(f'{f}: produced by the final_fit stage')